### Team Style Report

The goal of this notebook is to pull more data from the NBA API and create a 1 page pdf with important initial metrics and ranks.
This can then be used to summarize a team's strengths and weaknesses and understand some more of their style and tendencies.

## 1. Install / Imports

In [179]:
import time
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.patches import Rectangle
import textwrap
import urllib.request
import requests
from io import BytesIO
from PIL import Image

from nba_api.stats.static import teams
from nba_api.stats.endpoints import (
    leaguedashteamstats,
    leaguedashteamshotlocations,
    synergyplaytypes,
    leaguedashptteamdefend,
    leaguehustlestatsteam,
)

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)

## 2. User Selections

Select team, season, season type, and stat type.

In [ ]:
TEAM_NAME = "Los Angeles Clippers"
SEASON = "2025-26"
SEASON_TYPE = "Regular Season"
PER_MODE = "Per100Possessions"

## 3. Team Lookup and Colors

Make sure the team exists, then define and find their colors.

In [181]:
NBA_TEAMS = teams.get_teams()
TEAM_LOOKUP = {t["full_name"]: t for t in NBA_TEAMS}
TEAM_ABBR_LOOKUP = {t["abbreviation"]: t for t in NBA_TEAMS}

TEAM_COLORS = {
    "ATL": ("#E03A3E", "#C1D32F", "https://en.wikipedia.org/wiki/Special:FilePath/Atlanta_Hawks_logo.svg?width=512"),
    "BOS": ("#007A33", "#BA9653", "https://en.wikipedia.org/wiki/Special:FilePath/Boston_Celtics.svg?width=512"),
    "BKN": ("#000000", "#FFFFFF", "https://en.wikipedia.org/wiki/Special:FilePath/Brooklyn_Nets_newlogo.svg?width=512"),
    "CHA": ("#1D1160", "#00788C", "https://en.wikipedia.org/wiki/Special:FilePath/Charlotte_Hornets_(2014).svg?width=512"),
    "CHI": ("#CE1141", "#000000", "https://en.wikipedia.org/wiki/Special:FilePath/Chicago_Bulls_logo.svg?width=512"),
    "CLE": ("#860038", "#FDBB30", "https://en.wikipedia.org/wiki/Special:FilePath/Cleveland_Cavaliers_logo.svg?width=512"),
    "DAL": ("#00538C", "#002B5E", "https://en.wikipedia.org/wiki/Special:FilePath/Dallas_Mavericks_logo.svg?width=512"),
    "DEN": ("#0E2240", "#FEC524", "https://en.wikipedia.org/wiki/Special:FilePath/Denver_Nuggets.svg?width=512"),
    "DET": ("#C8102E", "#1D42BA", "https://en.wikipedia.org/wiki/Special:FilePath/Logo_of_the_Detroit_Pistons.svg?width=512"),
    "GSW": ("#1D428A", "#FFC72C", "https://en.wikipedia.org/wiki/Special:FilePath/Golden_State_Warriors_logo.svg?width=512"),
    "HOU": ("#CE1141", "#000000", "https://en.wikipedia.org/wiki/Special:FilePath/Houston_Rockets.svg?width=512"),
    "IND": ("#002D62", "#FDBB30", "https://en.wikipedia.org/wiki/Special:FilePath/Indiana_Pacers.svg?width=512"),
    "LAC": ("#C8102E", "#1D428A", "https://en.wikipedia.org/wiki/Special:FilePath/Los_Angeles_Clippers_(2024).svg?width=512"),
    "LAL": ("#552583", "#FDB927", "https://en.wikipedia.org/wiki/Special:FilePath/Los_Angeles_Lakers_logo.svg?width=512"),
    "MEM": ("#5D76A9", "#12173F", "https://en.wikipedia.org/wiki/Special:FilePath/Memphis_Grizzlies.svg?width=512"),
    "MIA": ("#98002E", "#F9A01B", "https://en.wikipedia.org/wiki/Special:FilePath/Miami_Heat_logo.svg?width=512"),
    "MIL": ("#00471B", "#EEE1C6", "https://en.wikipedia.org/wiki/Special:FilePath/Milwaukee_Bucks_logo.svg?width=512"),
    "MIN": ("#0C2340", "#78BE20", "https://en.wikipedia.org/wiki/Special:FilePath/Minnesota_Timberwolves_logo.svg?width=512"),
    "NOP": ("#0C2340", "#C8102E", "https://en.wikipedia.org/wiki/Special:FilePath/New_Orleans_Pelicans_logo.svg?width=512"),
    "NYK": ("#006BB6", "#F58426", "https://en.wikipedia.org/wiki/Special:FilePath/New_York_Knicks_logo.svg?width=512"),
    "OKC": ("#007AC1", "#EF3B24", "https://en.wikipedia.org/wiki/Special:FilePath/Oklahoma_City_Thunder.svg?width=512"),
    "ORL": ("#0077C0", "#000000", "https://en.wikipedia.org/wiki/Special:FilePath/Orlando_Magic_logo.svg?width=512"),
    "PHI": ("#006BB6", "#ED174C", "https://en.wikipedia.org/wiki/Special:FilePath/Philadelphia_76ers_logo.svg?width=512"),
    "PHX": ("#1D1160", "#E56020", "https://en.wikipedia.org/wiki/Special:FilePath/Phoenix_Suns_logo.svg?width=512"),
    "POR": ("#E03A3E", "#000000", "https://en.wikipedia.org/wiki/Special:FilePath/Portland_Trail_Blazers_logo.svg?width=512"),
    "SAC": ("#5A2D81", "#63727A", "https://en.wikipedia.org/wiki/Special:FilePath/SacramentoKings.svg?width=512"),
    "SAS": ("#C4CED4", "#000000", "https://en.wikipedia.org/wiki/Special:FilePath/San_Antonio_Spurs.svg?width=512"),
    "TOR": ("#CE1141", "#000000", "https://en.wikipedia.org/wiki/Special:FilePath/Toronto_Raptors_logo.svg?width=512"),
    "UTA": ("#002B5C", "#F9A01B", "https://en.wikipedia.org/wiki/Special:FilePath/Utah_Jazz_logo_2022.svg?width=512"),
    "WAS": ("#002B5C", "#E31837", "https://en.wikipedia.org/wiki/Special:FilePath/Washington_Wizards_logo.svg?width=512"),
}

def get_team(team_name):
    if team_name not in TEAM_LOOKUP:
        matches = [name for name in TEAM_LOOKUP if team_name.lower() in name.lower()]
        raise ValueError(f"Team not found. Close matches: {matches}")
    return TEAM_LOOKUP[team_name]

TEAM = get_team(TEAM_NAME)
TEAM_ID = TEAM["id"]
TEAM_ABBR = TEAM["abbreviation"]
PRIMARY, SECONDARY, TEAM_LOGO_URL = TEAM_COLORS.get(TEAM_ABBR, ("#222222", "#888888", None))

## 4. Functions to Help With the API

In [182]:
def fetch_with_retry(endpoint_func, max_retries=3, sleep=1.5, **kwargs):
    last_error = None

    for attempt in range(max_retries):
        try:
            result = endpoint_func(timeout=60, **kwargs)
            time.sleep(0.6)
            return result.get_data_frames()
        except Exception as e:
            last_error = e
            time.sleep(sleep * (attempt + 1))

    raise last_error


def rank_col(series, ascending=False):
    return series.rank(method="min", ascending=ascending).astype("Int64")


def clean_rank_value(x):
    if pd.isna(x):
        return "N/A"
    return f"{int(x)}"


def pct(x, digits=1):
    if pd.isna(x):
        return "N/A"
    return f"{x * 100:.{digits}f}%"


def num(x, digits=1):
    if pd.isna(x):
        return "N/A"
    return f"{x:.{digits}f}"

## 5. Pull Core Team Stats

In [183]:
def get_team_stats(measure_type, per_mode=PER_MODE):
    df = fetch_with_retry(
        leaguedashteamstats.LeagueDashTeamStats,
        season=SEASON,
        season_type_all_star=SEASON_TYPE,
        measure_type_detailed_defense=measure_type,
        per_mode_detailed=PER_MODE,
        rank="Y",
        pace_adjust="N",
        plus_minus="N",
        league_id_nullable="00",
        last_n_games=0,
        month=0,
        opponent_team_id=0,
        period=0,
    )[0]
    return df

base_df = get_team_stats("Base")
adv_df = get_team_stats("Advanced")
four_df = get_team_stats("Four Factors")
misc_df = get_team_stats("Misc")
scoring_df = get_team_stats("Scoring")
opp_df = get_team_stats("Opponent")
def_df = get_team_stats("Defense")

team_base = base_df.loc[base_df["TEAM_ID"] == TEAM_ID].iloc[0]
team_adv = adv_df.loc[adv_df["TEAM_ID"] == TEAM_ID].iloc[0]
team_four = four_df.loc[four_df["TEAM_ID"] == TEAM_ID].iloc[0]
team_misc = misc_df.loc[misc_df["TEAM_ID"] == TEAM_ID].iloc[0]
team_scoring = scoring_df.loc[scoring_df["TEAM_ID"] == TEAM_ID].iloc[0]
team_opp = opp_df.loc[opp_df["TEAM_ID"] == TEAM_ID].iloc[0]
team_def = def_df.loc[def_df["TEAM_ID"] == TEAM_ID].iloc[0]

## 6. Synergy Play Types

In [184]:
PLAY_TYPES = [
    "Transition",
    "Isolation",
    "PRBallHandler",
    "PRRollMan",
    "Postup",
    "Spotup",
    "Handoff",
    "Cut",
    "OffScreen",
    "OffRebound",
    "Misc",
]

PLAY_TYPE_LABELS = { 
    "Transition": "Transition",
    "Isolation": "Isolation",
    "PRBallHandler": "P&R Ball Handler",
    "PRRollMan": "P&R Roll Man",
    "Postup": "Post-Up",
    "Spotup": "Spot-Up",
    "Handoff":  "Handoff",
    "Cut": "Cut",
    "OffScreen": "Off Screens",
    "OffRebound": "Put Backs",
    "Misc": "Misc",
}

def get_synergy(grouping="Offensive"):
    frames = []
    for play_type in PLAY_TYPES:
        try:
            df = fetch_with_retry(
                synergyplaytypes.SynergyPlayTypes,
                league_id="00",
                per_mode_simple="Totals",
                player_or_team_abbreviation="T",
                season=SEASON,
                season_type_all_star=SEASON_TYPE,
                play_type_nullable=play_type,
                type_grouping_nullable=grouping,
            )[0]
            if len(df):
                df["PLAY_TYPE_RAW"] = play_type
                df["PLAY_TYPE_LABEL"] = PLAY_TYPE_LABELS[play_type]
                df["GROUPING"] = grouping
                frames.append(df)
        except Exception as e:
            print(f"Skipped {grouping} {play_type}: {e}")
    if not frames:
        return pd.DataFrame()
    out = pd.concat(frames, ignore_index=True)
    out["POSS_PCT_RANK"] = out.groupby("PLAY_TYPE_RAW")["POSS_PCT"].rank(
        ascending=False, method="min"
    ).astype("Int64")
    out["PPP_RANK"] = out.groupby("PLAY_TYPE_RAW")["PPP"].rank(
        ascending=False if grouping == "offensive" else True,
        method="min"
    ).astype("Int64")
    return out

synergy_off = get_synergy("Offensive")
synergy_def = get_synergy("Defensive")

team_synergy_off = synergy_off.loc[synergy_off["TEAM_ID"] == TEAM_ID].copy()
team_synergy_def = synergy_def.loc[synergy_def["TEAM_ID"] == TEAM_ID].copy()

## 7. Shot Locations

In [185]:
SHOT_ZONE_LABELS = [
    "Restricted Area",
    "In The Paint (Non-RA)",
    "Mid-Range",
    "Left Corner 3",
    "Right Corner 3",
    "Above the Break 3",
    "Backcourt"
]

def get_team_shot_locations(measure_type="Base"):
    df = fetch_with_retry(
        leaguedashteamshotlocations.LeagueDashTeamShotLocations,
        season=SEASON,
        season_type_all_star=SEASON_TYPE,
        measure_type_simple=measure_type,
        per_mode_detailed="Totals",
        distance_range="By Zone",
        rank="N",
        pace_adjust="N",
        plus_minus="N",
        league_id_nullable="00",
        last_n_games=0,
        month=0,
        opponent_team_id=0,
        period=0,
    )[0]
    rows = []
    for _, row in df.iterrows():
        team_id = row.iloc[0]
        team_name = row.iloc[1]
        for i, zone in enumerate(SHOT_ZONE_LABELS):
            start = 2 + i*3
            rows.append({
                "TEAM_ID": team_id,
                "TEAM_NAME": team_name,
                "ZONE": zone,
                "FGM": row.iloc[start],
                "FGA": row.iloc[start+1],
                "FG_PCT": row.iloc[start+2],
            })
    tidy = pd.DataFrame(rows)
    totals = tidy.groupby("TEAM_ID")["FGA"].sum().rename("TOTAL_FGA")
    tidy = tidy.merge(totals, on="TEAM_ID", how="left")
    tidy["FGA_FREQ"] = tidy["FGA"] / tidy["TOTAL_FGA"]
    tidy["FGA_FREQ_RANK"] = tidy.groupby("ZONE")["FGA_FREQ"].rank(
        ascending=False, method="min"
    ).astype("Int64")
    tidy["FG_PCT_RANK"] = tidy.groupby("ZONE")["FG_PCT"].rank(
        ascending=False, method="min"
    ).astype("Int64")
    return tidy

shot_locations = get_team_shot_locations("Base")
opp_shot_locations = get_team_shot_locations("Opponent")

team_shots = shot_locations.loc[shot_locations["TEAM_ID"] == TEAM_ID].copy()
team_opp_shots = opp_shot_locations.loc[opp_shot_locations["TEAM_ID"] == TEAM_ID].copy()

## 8. Defensive Shot Profile + Hustle

In [186]:
DEFENSE_CATEGORIES = [
    "Overall",
    "3 Pointers",
    "2 Pointers",
    "Less Than 6Ft",
    "Less Than 10Ft",
    "Greater Than 15 Ft",
]

def get_defended_shots():
    frames = []
    for cat in DEFENSE_CATEGORIES:
        try:
            df = fetch_with_retry(
                leaguedashptteamdefend.LeagueDashPtTeamDefend,
                defense_category=cat,
                league_id="00",
                per_mode_simple="Totals",
                season=SEASON,
                season_type_all_star=SEASON_TYPE,
            )[0]
            df["DEFENSE_CATEGORY"] = cat
            frames.append(df)
        except Exception as e:
            print(f"Skipped defense category {cat}: {e}")
    out = pd.concat(frames, ignore_index=True)
    out["D_FG_PCT_RANK"] = out.groupby("DEFENSE_CATEGORY")["D_FG_PCT"].rank(
        ascending=True, method="min"
    ).astype("Int64")
    out["PCT_PLUSMINUS_RANK"] = out.groupby("DEFENSE_CATEGORY")["PCT_PLUSMINUS"].rank(
        ascending=True, method="min"
    ).astype("Int64")
    return out

def get_hustle():
    df = fetch_with_retry(
        leaguehustlestatsteam.LeagueHustleStatsTeam,
        season=SEASON,
        season_type_all_star=SEASON_TYPE,
        per_mode_time="PerGame",
        league_id_nullable="00",
    )[0]
    for col in ["CONTESTED_SHOTS", "DEFLECTIONS", "CHARGES_DRAWN", "SCREEN_ASSISTS", "LOOSE_BALLS_RECOVERED", "BOX_OUTS"]:
        if col in df.columns:
            df[f"{col}_RANK"] = df[col].rank(ascending=False, method="min").astype("Int64")
    return df

defended = get_defended_shots()
hustle = get_hustle()

team_defended = defended.loc[defended["TEAM_ID"] == TEAM_ID].copy()
team_hustle = hustle.loc[hustle["TEAM_ID"] == TEAM_ID].copy()

Skipped defense category Greater Than 15 Ft: 'resultSet'


## 9. Summary Tables

In [187]:
summary_rows = [
    ("Record", f"{int(team_adv['W'])}-{int(team_adv['L'])}", clean_rank_value(team_base.get("W_PCT_RANK"))),
    ("Net Rating", num(team_adv.get("NET_RATING")), clean_rank_value(team_adv.get("NET_RATING_RANK"))),
    ("Off Rating", num(team_adv.get("OFF_RATING")), clean_rank_value(team_adv.get("OFF_RATING_RANK"))),
    ("Def Rating", num(team_adv.get("DEF_RATING")), clean_rank_value(team_adv.get("DEF_RATING_RANK"))),
    ("Pace", num(team_adv.get("PACE")), clean_rank_value(team_adv.get("PACE_RANK"))),
    ("Assist %", num(team_adv.get("AST_PCT")*100), clean_rank_value(team_adv.get("AST_PCT_RANK"))),
    ("Rebound %", num(team_adv.get("REB_PCT")*100), clean_rank_value(team_adv.get("REB_PCT_RANK"))),
    ("True Shooting %", num(team_adv.get("TS_PCT")*100), clean_rank_value(team_adv.get("TS_PCT_RANK"))),
]
summary_table = pd.DataFrame(summary_rows, columns=["Metirc", "Value", "Rank"])

four_factor_metrics = [
    ("eFG%", "EFG_PCT"),
    ("FTA Rate", "FTA_RATE"),
    ("TOV%", "TM_TOV_PCT"),
    ("OREB%", "OREB_PCT"),
    ("Opp eFG%", "OPP_EFG_PCT"),
    ("Opp FTA Rate", "OPP_FTA_RATE"),
    ("Opp TOV%", "OPP_TOV_PCT"),
    ("DREB%", "OPP_OREB_PCT"),
]
four_table_rows = []
for label, col in four_factor_metrics:
    val = team_four.get(col, np.nan)
    rank = team_four.get(col, np.nan)
    four_table_rows.append((label, pct(val), clean_rank_value(rank)))

four_table = pd.DataFrame(four_table_rows, columns=["Four Factor", "Value", "Rank"])

four_factor_values = [
    team_adv["EFG_PCT"],
    (team_base["FTM"])/(team_base["FGA"]),
    team_adv["TM_TOV_PCT"],
    team_adv["OREB_PCT"],
    ((team_opp["OPP_FGM"])+(team_opp["OPP_FG3M"]))/(team_opp["OPP_FGA"]),
    (team_opp["OPP_FTM"])/(team_opp["OPP_FGA"]),
    (team_opp["OPP_TOV"])/((team_opp["OPP_TOV"])+(0.44*(team_opp["OPP_FTA"]))+(team_opp["OPP_FGA"])),
    team_adv["DREB_PCT"],
]
four_table["Value"] = [pct(x) for x in four_factor_values]

play_type_table = team_synergy_off[["PLAY_TYPE_LABEL", "POSS_PCT", "POSS_PCT_RANK", "PPP", "PPP_RANK"]].copy()
play_type_table["POSS_PCT"] = [pct(x) for x in play_type_table["POSS_PCT"]]

def_play_type_table = team_synergy_def[["PLAY_TYPE_LABEL", "POSS_PCT", "POSS_PCT_RANK", "PPP", "PPP_RANK"]].copy()
def_play_type_table["POSS_PCT"] = [pct(x) for x in def_play_type_table["POSS_PCT"]]

shot_loc_table = team_shots[["ZONE", "FGA_FREQ", "FGA_FREQ_RANK", "FG_PCT", "FG_PCT_RANK"]]
shot_loc_table["FGA_FREQ"] = [pct(x) for x in shot_loc_table["FGA_FREQ"]]
shot_loc_table["FG_PCT"] = [pct(x) for x in shot_loc_table["FG_PCT"]]
shot_loc_table = shot_loc_table[shot_loc_table["ZONE"] != "Backcourt"].copy()

opp_shot_loc_table = team_opp_shots[["ZONE", "FGA_FREQ", "FGA_FREQ_RANK", "FG_PCT", "FG_PCT_RANK"]]
opp_shot_loc_table["FGA_FREQ"] = [pct(x) for x in opp_shot_loc_table["FGA_FREQ"]]
opp_shot_loc_table["FG_PCT"] = [pct(x) for x in opp_shot_loc_table["FG_PCT"]]
opp_shot_loc_table = opp_shot_loc_table[opp_shot_loc_table["ZONE"] != "Backcourt"].copy()

misc_metrics = [
    ("PPG off TOV", "PTS_OFF_TOV"),
    ("2nd Chance PPG", "PTS_2ND_CHANCE"),
    ("Fastbreak PPG", "PTS_FB"),
    ("PPG in Paint", "PTS_PAINT")
]
misc_rows = []
for label, col in misc_metrics:
    val = team_misc.get(col, np.nan)
    rank = team_misc.get(f"{col}_RANK", np.nan)
    misc_rows.append((label, val, clean_rank_value(rank)))
off_misc = pd.DataFrame(misc_rows, columns=["Metric", "Value", "Rank"])

def_misc_metrics = [
    ("PPG off TOV", "OPP_PTS_OFF_TOV"),
    ("2nd Chance PPG", "OPP_PTS_2ND_CHANCE"),
    ("Fastbreak PPG", "OPP_PTS_FB"),
    ("PPG in Paint", "OPP_PTS_PAINT")
]
def_misc_rows = []
for label, col in def_misc_metrics:
    val = team_misc.get(col, np.nan)
    rank = team_misc.get(f"{col}_RANK", np.nan)
    def_misc_rows.append((label, val, clean_rank_value(rank)))
def_misc = pd.DataFrame(def_misc_rows, columns=["Metric", "Value", "Rank"])

## 10. One-Page Portrait PDF Helpers

In [188]:
def load_logo_image(url):
    if not url:
        return None
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=20) as response:
            return Image.open(BytesIO(response.read())).convert("RGBA")
    except Exception as e:
        print(f"Logo unavailable: {e}")
        return None


def rank_color(value):
    try:
        rank = int(float(value))
    except Exception:
        return None

    if 1 <= rank <= 10:
        return "#D9F2E3"
    if 11 <= rank <= 20:
        return "#E5E7EB"
    if 21 <= rank <= 30:
        return "#F8D7DA"
    return None


def clean_table_for_pdf(df):
    out = df.copy()

    rename_map = {
        "Metirc": "Metric",
        "PLAY_TYPE_LABEL": "Play Type",
        "POSS_PCT": "Poss%",
        "POSS_PCT_RANK": "Poss Rank",
        "PPP_RANK": "PPP Rank",
        "FGA_FREQ": "Freq",
        "FGA_FREQ_RANK": "Freq Rank",
        "FG_PCT": "FG%",
        "FG_PCT_RANK": "FG% Rank",
        "ZONE": "Zone",
    }

    out = out.rename(columns={c: rename_map.get(c, c) for c in out.columns})

    for col in out.columns:
        if "Rank" in col:
            out[col] = out[col].apply(
                lambda x: "" if pd.isna(x) else str(int(float(x))) if str(x).replace(".", "", 1).isdigit() else str(x)
            )

        if col == "PPP":
            out[col] = out[col].apply(
                lambda x: "" if pd.isna(x) else f"{float(x):.3f}" if isinstance(x, (int, float, np.number)) else x
            )

        if col == "Value":
            out[col] = out[col].apply(
                lambda x: "" if pd.isna(x) else f"{float(x):.1f}" if isinstance(x, (int, float, np.number)) else x
            )

    return out


def wrap_cells(df, width=22):
    out = df.copy()
    for col in out.columns:
        out[col] = out[col].astype(str).apply(
            lambda x: "\n".join(
                textwrap.wrap(x, width=width, break_long_words=False)
            ) if len(x) > width else x
        )
    return out


def make_four_factor_grouped_table(df):
    out = clean_table_for_pdf(df)

    first_col = out.columns[0]
    out[first_col] = (
        out[first_col]
        .astype(str)
        .str.replace("Opp ", "", regex=False)
        .str.replace("Opponent ", "", regex=False)
    )

    offense = pd.DataFrame([["Offense"] + [""] * (len(out.columns) - 1)], columns=out.columns)
    defense = pd.DataFrame([["Defense"] + [""] * (len(out.columns) - 1)], columns=out.columns)

    return pd.concat(
        [offense, out.iloc[:4], defense, out.iloc[4:]],
        ignore_index=True
    )


def draw_pdf_table(
    ax,
    df,
    title,
    title_color,
    header_color,
    font_size=6.4,
    first_col_width=0.38,
    wrap_width=22,
    is_four_factors=False,
    table_height=0.890,
):
    ax.axis("off")

    if is_four_factors:
        df = make_four_factor_grouped_table(df)
    else:
        df = clean_table_for_pdf(df)

    df = wrap_cells(df, width=wrap_width)

    # Fixed title/table spacing for every table.
    ax.text(
        0.0,
        0.985,
        title.upper(),
        transform=ax.transAxes,
        fontsize=10.1,
        fontweight="bold",
        fontfamily="DejaVu Sans",
        color=title_color,
        va="top",
        ha="left",
    )

    n_cols = len(df.columns)
    other_width = (1 - first_col_width) / max(n_cols - 1, 1)
    col_widths = [first_col_width] + [other_width] * (n_cols - 1)

    table_bbox = [0, 0, 1, table_height]

    table = ax.table(
        cellText=df.values,
        colLabels=df.columns,
        cellLoc="center",
        colLoc="center",
        colWidths=col_widths,
        bbox=table_bbox,
    )

    table.auto_set_font_size(False)
    table.set_fontsize(font_size)

    group_rows = [1, 6] if is_four_factors else []
    group_fill = "#E6F0FB"
    group_edge = "#E6F0FB"
    n_table_rows = len(df) + 1

    for (row, col), cell in table.get_celld().items():
        cell.set_linewidth(0.25)
        cell.set_edgecolor("#D8DEE6")

        if row == 0:
            cell.set_facecolor(header_color)
            cell.set_text_props(color="white", weight="bold")
            continue

        if row in group_rows:
            cell.set_facecolor(group_fill)
            cell.set_edgecolor(group_edge)
            cell.set_linewidth(0.0)
            cell.visible_edges = "closed"
            cell.get_text().set_text("")
            continue

        cell.set_facecolor("#FFFFFF" if row % 2 else "#F6F8FA")

        if col == 0:
            cell.set_text_props(ha="left", color="#1F2933")
        else:
            cell.set_text_props(color="#1F2933")

        col_name = df.columns[col]
        if "Rank" in col_name:
            face = rank_color(cell.get_text().get_text())
            if face:
                cell.get_text().set_bbox({
                    "boxstyle": "round,pad=0.23",
                    "facecolor": face,
                    "edgecolor": "#9CA3AF",
                    "linewidth": 0.45,
                })

    if is_four_factors:
        bbox_y, bbox_h = table_bbox[1], table_bbox[3]
        for row, label in [(1, "Offense"), (6, "Defense")]:
            y_center = bbox_y + bbox_h * (1 - (row + 0.5) / n_table_rows)
            ax.text(
                0.5,
                y_center,
                label,
                transform=ax.transAxes,
                ha="center",
                va="center",
                fontsize=font_size + 1.5,
                fontweight="bold",
                color=PRIMARY,
                zorder=10,
            )

    return table

## 11. Table Ordering & Trimming

In [189]:
def percent_to_float(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, str):
        return float(x.replace("%", "").strip())
    return float(x) * 100 if x <= 1 else float(x)


play_type_pdf = play_type_table.copy()
def_play_type_pdf = def_play_type_table.copy()
shot_loc_pdf = shot_loc_table.copy()
opp_shot_loc_pdf = opp_shot_loc_table.copy()

if "POSS_PCT" in play_type_pdf.columns:
    play_type_pdf = play_type_pdf.sort_values("POSS_PCT", key=lambda s: s.map(percent_to_float), ascending=False)
elif "Poss%" in play_type_pdf.columns:
    play_type_pdf = play_type_pdf.sort_values("Poss%", key=lambda s: s.map(percent_to_float), ascending=False)

if "POSS_PCT" in def_play_type_pdf.columns:
    def_play_type_pdf = def_play_type_pdf.sort_values("POSS_PCT", key=lambda s: s.map(percent_to_float), ascending=False)
elif "Poss%" in def_play_type_pdf.columns:
    def_play_type_pdf = def_play_type_pdf.sort_values("Poss%", key=lambda s: s.map(percent_to_float), ascending=False)

## 12. Create the One-Page Portrait PDF

In [190]:
OUTPUT_DIR = Path("team_style_1page_pdf_reports")
OUTPUT_DIR.mkdir(exist_ok=True)

pdf_path = OUTPUT_DIR / f"{TEAM_ABBR}_{SEASON}_TS_report.pdf"
logo_img = load_logo_image(TEAM_LOGO_URL)

with PdfPages(pdf_path) as pdf:
    fig = plt.figure(figsize=(8.5, 11))
    fig.patch.set_facecolor("#FFFFFF")

    # Header
    header_ax = fig.add_axes([0.045, 0.91, 0.91, 0.065])
    header_ax.axis("off")
    header_ax.add_patch(
        Rectangle((0, 0), 1, 1, transform=header_ax.transAxes, color=PRIMARY)
    )

    header_ax.text(
        0.025, 0.62, TEAM_NAME,
        color="white", fontsize=20, fontweight="bold",
        va="center", ha="left",
    )

    header_ax.text(
        0.025, 0.24,
        f"Team Style Report | {SEASON} {SEASON_TYPE}",
        color=SECONDARY, fontsize=8.5, fontweight="bold",
        va="center", ha="left",
    )

    if logo_img is not None:
        logo_ax = fig.add_axes([0.842, 0.914, 0.09, 0.058])
        logo_ax.axis("off")
        logo_ax.imshow(logo_img)
    else:
        header_ax.text(
            0.975, 0.50, TEAM_ABBR,
            color="white", fontsize=22, fontweight="bold",
            va="center", ha="right", alpha=0.95,
        )

    left_x = 0.055
    right_x = 0.525
    col_w = 0.42

    y_summary = 0.745
    y_label = 0.692
    y_play = 0.455
    y_shots = 0.275
    y_misc = 0.102

    h_summary = 0.140
    h_label = 0.038
    h_play = 0.225
    h_shots = 0.158
    h_misc = 0.145

    fig.patches.extend([
        Rectangle(
            (left_x - 0.012, 0.075), col_w + 0.024, 0.650,
            transform=fig.transFigure,
            facecolor="#F7FBFF",
            edgecolor="#D8E8F8",
            linewidth=0.9,
            zorder=-1,
        ),
        Rectangle(
            (right_x - 0.012, 0.075), col_w + 0.024, 0.650,
            transform=fig.transFigure,
            facecolor="#FFF8F8",
            edgecolor="#F1D6D6",
            linewidth=0.9,
            zorder=-1,
        ),
    ])

    for x, label, fill, edge in [
        (left_x, "OFFENSE", "#DCEEFF", "#9CC5EA"),
        (right_x, "DEFENSE", "#FFE3E3", "#E4A6A6"),
    ]:
        label_ax = fig.add_axes([x - 0.012, y_label, col_w + 0.024, h_label])
        label_ax.axis("off")
        label_ax.add_patch(
            Rectangle(
                (0, 0), 1, 1,
                transform=label_ax.transAxes,
                facecolor=fill,
                edgecolor=edge,
                linewidth=0.9,
            )
        )
        label_ax.text(
            0.5, 0.5, label,
            fontsize=14.5,
            fontweight="black",
            fontfamily="DejaVu Sans",
            color=PRIMARY,
            ha="center",
            va="center",
        )

    ax_summary = fig.add_axes([left_x, y_summary, col_w, h_summary])
    draw_pdf_table(
        ax_summary, summary_table, "Team Snapshot",
        PRIMARY, PRIMARY,
        font_size=7.2,
        first_col_width=0.48,
        wrap_width=22,
        table_height=0.890,
    )

    ax_four = fig.add_axes([right_x, y_summary, col_w, h_summary])
    draw_pdf_table(
        ax_four, four_table, "Four Factors",
        PRIMARY, PRIMARY,
        font_size=6.55,
        first_col_width=0.42,
        wrap_width=18,
        is_four_factors=True,
        table_height=0.890,
    )

    ax_play = fig.add_axes([left_x, y_play, col_w, h_play])
    draw_pdf_table(
        ax_play, play_type_pdf, "Play Types",
        PRIMARY, PRIMARY,
        font_size=6.05,
        first_col_width=0.39,
        wrap_width=20,
        table_height=0.915,
    )

    ax_def_play = fig.add_axes([right_x, y_play, col_w, h_play])
    draw_pdf_table(
        ax_def_play, def_play_type_pdf, "Play Types",
        PRIMARY, PRIMARY,
        font_size=6.05,
        first_col_width=0.39,
        wrap_width=20,
        table_height=0.915,
    )

    ax_shots = fig.add_axes([left_x, y_shots, col_w, h_shots])
    draw_pdf_table(
        ax_shots, shot_loc_pdf, "Shot Locations",
        PRIMARY, PRIMARY,
        font_size=5.95,
        first_col_width=0.46,
        wrap_width=24,
        table_height=0.890,
    )

    ax_opp_shots = fig.add_axes([right_x, y_shots, col_w, h_shots])
    draw_pdf_table(
        ax_opp_shots, opp_shot_loc_pdf, "Shot Locations",
        PRIMARY, PRIMARY,
        font_size=5.95,
        first_col_width=0.46,
        wrap_width=24,
        table_height=0.890,
    )

    ax_off_misc = fig.add_axes([left_x, y_misc, col_w, h_misc])
    draw_pdf_table(
        ax_off_misc, off_misc, "Miscellaneous",
        PRIMARY, PRIMARY,
        font_size=6.45,
        first_col_width=0.55,
        wrap_width=22,
        table_height=0.890,
    )

    ax_def_misc = fig.add_axes([right_x, y_misc, col_w, h_misc])
    draw_pdf_table(
        ax_def_misc, def_misc, "Miscellaneous",
        PRIMARY, PRIMARY,
        font_size=6.45,
        first_col_width=0.55,
        wrap_width=22,
        table_height=0.890,
    )

    fig.text(
        0.055, 0.025,
        "Data: NBA Stats API / nba_api.",
        fontsize=6.5,
        color="#6B7280",
        ha="left",
    )

    pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)

print(f"Saved PDF: {pdf_path}")

Saved PDF: team_style_1page_pdf_reports/OKC_2024-25_TS_report.pdf
